# TP FIA4 — Harmonisation et qualité des données de santé

**Objectif :** harmoniser les trois fichiers hospitaliers, contrôler la qualité, produire le CSV harmonisé, le schéma cible, le profil de qualité et un rapport descriptif.



In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

try:
    import statsmodels.api as sm
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("Statsmodels disponible :", STATSMODELS_AVAILABLE)

Pandas : 3.0.6
NumPy : 2.4.4
Statsmodels disponible : True


In [2]:
# Détecter automatiquement où se trouvent les trois CSV.
# Fonctionne si le notebook est dans le dossier parent OU dans le dossier contenant les CSV.
candidates = [Path.cwd(), Path.cwd() / "sujet", Path.cwd().parent / "sujet"]
INPUT_DIR = next(
    (p for p in candidates if all((p / f"hopital_{h}.csv").exists() for h in ["A", "B", "C"])),
    None
)
if INPUT_DIR is None:
    raise FileNotFoundError(
        "Impossible de trouver hopital_A.csv, hopital_B.csv et hopital_C.csv. "
        "Placez le notebook dans le dossier parent de 'sujet' ou dans le dossier contenant les CSV."
    )
OUTPUT_DIR = Path.cwd() / "resultats"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Dossier des données :", INPUT_DIR.resolve())
print("Dossier de sortie :", OUTPUT_DIR.resolve())
print("Fichiers trouvés :", sorted(p.name for p in INPUT_DIR.glob("*.csv")))

Dossier des données : C:\Users\HP\Downloads\sujet (1)\sujet
Dossier de sortie : C:\Users\HP\Downloads\sujet (1)\sujet\resultats
Fichiers trouvés : ['hopital_A.csv', 'hopital_B.csv', 'hopital_C.csv']


## 1. Fonctions d'harmonisation

In [3]:
def read_csv(path):
    # Lire les identifiants en texte pour éviter de perdre les zéros initiaux.
    return pd.read_csv(path, dtype=str)

def yes_no(series):
    x = series.astype("string").str.strip().str.lower()
    mapping = {
        "yes": 1, "y": 1, "oui": 1, "1": 1, "1.0": 1, "true": 1,
        "no": 0, "n": 0, "non": 0, "0": 0, "0.0": 0, "false": 0
    }
    return pd.to_numeric(x.map(mapping), errors="coerce")

def sex_clean(series):
    x = series.astype("string").str.strip().str.lower()
    return x.map({
        "m": "M", "male": "M", "h": "M", "homme": "M",
        "f": "F", "female": "F", "femme": "F"
    })

def parse_dates(series, french=False):
    return pd.to_datetime(series, errors="coerce", dayfirst=french)

def num(series):
    # Accepte les virgules décimales éventuelles.
    return pd.to_numeric(series.astype("string").str.replace(",", ".", regex=False), errors="coerce")

def diagnosis_flags(series):
    x = series.astype("string").str.upper().fillna("")
    diabetes = x.str.contains(r"E11|DM2|DIABET", regex=True)
    hypertension = x.str.contains(r"I10|HTA|HYPERTENSION", regex=True)
    return diabetes, hypertension

In [10]:
def harmonize_a(path):
    d = read_csv(path)
    o = pd.DataFrame(index=d.index)

    o["patient_id"] = d["id"].astype("string").str.strip()
    o["hospital"] = "A"
    o["admission_date"] = parse_dates(d["date_adm"])
    o["discharge_date"] = parse_dates(d["date_dis"])
    o["age_years"] = num(d["age"])
    o["sex"] = sex_clean(d["gender"])
    o["weight_kg"] = num(d["weight"])
    o["height_cm"] = num(d["height_m"]) * 100
    o["smoking"] = yes_no(d["smoker"])
    o["diabetes"] = yes_no(d["diabetes"])
    o["hypertension"] = yes_no(d["hypertension"])
    o["icu"] = yes_no(d["icu"])
    o["los_reported_days"] = num(d["los"])
    o["readmit30"] = yes_no(d["readmit30"])
    o["in_hospital_death"] = yes_no(d["death"])
    o["diagnosis_raw"] = d["diag"].astype("string")
    o["sbp_mmhg"] = num(d["sbp"])
    o["creatinine_mg_dl"] = num(d["creatinine"])
    o["crp_mg_l"] = num(d["crp"])

    return o


def harmonize_b(path):
    d = read_csv(path)
    o = pd.DataFrame(index=d.index)

    o["patient_id"] = d["patient"].astype("string").str.strip()
    o["hospital"] = "B"
    o["admission_date"] = parse_dates(d["admit"], french=True)
    o["discharge_date"] = parse_dates(d["discharge"], french=True)
    o["age_years"] = num(d["age"])
    o["sex"] = sex_clean(d["gender"])
    o["weight_kg"] = num(d["weight_lb"]) * 0.45359237
    o["height_cm"] = num(d["height_cm"])
    o["smoking"] = yes_no(d["smoking"])
    o["diabetes"] = yes_no(d["diab"])
    o["hypertension"] = yes_no(d["htn"])
    o["icu"] = yes_no(d["icu"])
    o["los_reported_days"] = num(d["stay_days"])
    o["readmit30"] = yes_no(d["readmission"])
    o["in_hospital_death"] = yes_no(d["mortality"])
    o["diagnosis_raw"] = d["diagnosis"].astype("string")
    o["sbp_mmhg"] = num(d["blood_pressure"])
    o["creatinine_mg_dl"] = num(d["creat"])
    o["crp_mg_l"] = num(d["CRP"])

    return o


def harmonize_c(path):
    d = read_csv(path)
    o = pd.DataFrame(index=d.index)

    o["patient_id"] = d["id_patient"].astype("string").str.strip()
    o["hospital"] = "C"
    o["admission_date"] = parse_dates(d["date_entree"], french=True)
    o["discharge_date"] = parse_dates(d["date_sortie"], french=True)
    o["age_years"] = num(d["age"])
    o["sex"] = sex_clean(d["sexe"])
    o["weight_kg"] = num(d["poids"])

    # Convertir les tailles en mètres vers des centimètres.
    # Les tailles déjà exprimées en centimètres restent inchangées.
    height = pd.to_numeric(d["taille"], errors="coerce")
    height_cm = height.copy()
    mask_metres = height.notna() & (height < 3).fillna(False)
    height_cm.loc[mask_metres] = height.loc[mask_metres] * 100
    o["height_cm"] = height_cm

    o["smoking"] = yes_no(d["tabac"])
    o["diabetes"] = yes_no(d["diabete"])
    o["hypertension"] = yes_no(d["hta"])
    o["icu"] = yes_no(d["reanimation"])
    o["los_reported_days"] = num(d["duree"])
    o["readmit30"] = yes_no(d["rehosp_30j"])
    o["in_hospital_death"] = yes_no(d["deces"])
    o["diagnosis_raw"] = d["diag"].astype("string")
    o["sbp_mmhg"] = num(d["pas"])
    o["creatinine_mg_dl"] = num(d["creatininémie"])
    o["crp_mg_l"] = num(d["CRP"])

    return o

## 2. Explorer les sources avant nettoyage

In [11]:
source_paths = {h: INPUT_DIR / f"hopital_{h}.csv" for h in ["A", "B", "C"]}
raw_tables = {h: read_csv(p) for h, p in source_paths.items()}
source_summary = pd.DataFrame([
    {
        "hopital": h,
        "nombre_lignes": len(d),
        "nombre_colonnes": len(d.columns),
        "colonnes": ", ".join(d.columns)
    }
    for h, d in raw_tables.items()
])
display(source_summary)
for h, d in raw_tables.items():
    print(f"\nHôpital {h} — aperçu")
    display(d.head(3))

,hopital,nombre_lignes,nombre_colonnes,colonnes
0,A,3925,18,"id, date_adm, date_dis, age, gender, weight, h..."
1,B,3497,18,"patient, admit, discharge, age, gender, weight..."
2,C,2578,18,"id_patient, date_entree, date_sortie, age, sex..."



Hôpital A — aperçu


,id,date_adm,date_dis,age,gender,weight,height_m,smoker,diabetes,hypertension,icu,los,readmit30,death,diag,sbp,creatinine,crp
0,P100000,2025-07-10,2025-07-13,71.0,Male,150.57554599999997,1.73,NaN,no,no,1.0,3.2,0.0,0.0,I10,-1.0,0.52,NaN
1,P100001,2025-03-23,2025-03-26,60.0,Male,131.395352,1.66,no,no,no,0.0,3.3,0.0,0.0,I10,-1.0,0.88,NaN
2,P100008,2025-02-28,2025-03-12,54.0,Male,206.13197,1.64,no,yes,yes,0.0,12.6,0.0,0.0,E11.9+I10,-1.0,1.07,54.9



Hôpital B — aperçu


,patient,admit,discharge,age,gender,weight_lb,height_cm,smoking,diab,htn,icu,stay_days,readmission,mortality,diagnosis,blood_pressure,creat,CRP
0,P100002,22/01/2025,27/01/2025,74.0,M,84.00540682748047,180.0,N,0,1,0.0,5.3,0.0,0.0,DM2,999.0,1.02,26.9
1,P100003,03/03/2024,11/03/2024,89.0,M,48.71587847338771,163.0,Y,1,0,0.0,8.5,0.0,0.0,DM2,999.0,NaN,79.5
2,P100004,27/11/2025,02/12/2025,58.0,M,71.71303898177464,157.0,N,0,0,0.0,5.7,0.0,0.0,HTA,999.0,0.56,3.3



Hôpital C — aperçu


,id_patient,date_entree,date_sortie,age,sexe,poids,taille,tabac,diabete,hta,reanimation,duree,rehosp_30j,deces,diag,pas,creatininémie,CRP
0,P100010,2025/12/02,2025/12/05,54.0,Homme,67.1,1.84,NON,0,1,0.0,3.8,0.0,0.0,E11,35.0,1.14,5.6
1,P100012,2024/04/26,2024/04/27,66.0,Homme,90.2,1.88,NON,0,0,0.0,1.8,0.0,0.0,HTA,35.0,NaN,3.8
2,P100013,2024/07/10,2024/07/11,28.0,Homme,73.4,1.81,OUI,0,0,0.0,1.6,0.0,0.0,HTA,35.0,1.08,12.5


## 3. Harmoniser les colonnes et unités

In [12]:
# Harmoniser les trois hôpitaux et regrouper les données

df_a = harmonize_a(source_paths["A"])
df_b = harmonize_b(source_paths["B"])
df_c = harmonize_c(source_paths["C"])

# Combiner les données des trois hôpitaux
df = pd.concat(
    [df_a, df_b, df_c],
    ignore_index=True
)

# Afficher le nombre de patients par hôpital
print("Effectifs par hôpital :")
display(
    df.groupby("hospital")
      .size()
      .rename("n")
      .to_frame()
)

# Afficher les cinq premières lignes
display(df.head())

Effectifs par hôpital :


,n
hospital,
A,3925
B,3497
C,2578


,patient_id,hospital,admission_date,discharge_date,age_years,sex,weight_kg,height_cm,smoking,diabetes,hypertension,icu,los_reported_days,readmit30,in_hospital_death,diagnosis_raw,sbp_mmhg,creatinine_mg_dl,crp_mg_l
0,P100000,A,2025-07-10,2025-07-13,71.0,M,150.575546,173.0,NaN,0.0,0.0,1.0,3.2,0.0,0.0,I10,-1.0,0.52,<NA>
1,P100001,A,2025-03-23,2025-03-26,60.0,M,131.395352,166.0,0.0,0.0,0.0,0.0,3.3,0.0,0.0,I10,-1.0,0.88,<NA>
2,P100008,A,2025-02-28,2025-03-12,54.0,M,206.13197,164.0,0.0,1.0,1.0,0.0,12.6,0.0,0.0,E11.9+I10,-1.0,1.07,54.9
3,P100011,A,NaT,2025-08-28,54.0,M,193.345174,187.0,0.0,0.0,0.0,0.0,8.5,0.0,0.0,I10,-1.0,1.3,18.6
4,P100014,A,2024-03-03,2024-03-08,31.0,M,118.608556,175.0,0.0,0.0,1.0,0.0,5.0,0.0,NaN,E11.9,-1.0,1.16,18.8


## 4. Contrôler les valeurs aberrantes et les dates

In [13]:
# Les limites sont des règles de contrôle pédagogiques, à discuter avec les responsables des données.
ranges = {
    "age_years": (18, 110),
    "weight_kg": (25, 300),
    "height_cm": (100, 230),
    "sbp_mmhg": (60, 260),
    "creatinine_mg_dl": (0.1, 25),
    "crp_mg_l": (0, 1000),
    "los_reported_days": (0, 365),
}
invalid_counts = {}
for col, (low, high) in ranges.items():
    bad = df[col].notna() & ~df[col].between(low, high)
    invalid_counts[col] = int(bad.sum())
    df.loc[bad, col] = np.nan

bad_dates = (
    df["admission_date"].isna()
    | df["discharge_date"].isna()
    | (df["discharge_date"] < df["admission_date"])
)
date_incoherences = int(bad_dates.sum())
df.loc[bad_dates, ["admission_date", "discharge_date"]] = pd.NaT
df["los_days"] = (df["discharge_date"] - df["admission_date"]).dt.total_seconds() / 86400
df.loc[df["los_days"] < 0, "los_days"] = np.nan

# Si diabète/HTA est manquant, essayer de le déduire du diagnostic texte/code.
diag_diabetes, diag_htn = diagnosis_flags(df["diagnosis_raw"])
df["diabetes"] = df["diabetes"].fillna(diag_diabetes.astype(float))
df["hypertension"] = df["hypertension"].fillna(diag_htn.astype(float))
df["diagnosis_raw"] = df["diagnosis_raw"].str.strip().replace({"": pd.NA, "nan": pd.NA, "<NA>": pd.NA})

# Signaler les identifiants répétés, sans les supprimer automatiquement.
df["duplicate_patient_id"] = df["patient_id"].duplicated(keep=False)

# IMC calculé seulement lorsque taille et poids sont disponibles et plausibles.
df["bmi_kg_m2"] = df["weight_kg"] / (df["height_cm"] / 100) ** 2
df.loc[~df["bmi_kg_m2"].between(10, 80), "bmi_kg_m2"] = np.nan

print("Valeurs hors bornes remplacées par NA :")
display(pd.Series(invalid_counts, name="n_valeurs_invalides").to_frame())
print("Dates incohérentes ou manquantes :", date_incoherences)
print("Lignes avec identifiant dupliqué :", int(df["duplicate_patient_id"].sum()))

Valeurs hors bornes remplacées par NA :


,n_valeurs_invalides
age_years,0
weight_kg,5
height_cm,0
sbp_mmhg,30
creatinine_mg_dl,0
crp_mg_l,0
los_reported_days,0


Dates incohérentes ou manquantes : 2526
Lignes avec identifiant dupliqué : 0


## 5. Enregistrer le jeu harmonisé et le schéma cible

In [14]:
columns = [
    "patient_id", "hospital", "admission_date", "discharge_date", "age_years", "sex",
    "weight_kg", "height_cm", "bmi_kg_m2", "smoking", "diabetes", "hypertension", "icu",
    "los_days", "los_reported_days", "readmit30", "in_hospital_death", "diagnosis_raw",
    "sbp_mmhg", "creatinine_mg_dl", "crp_mg_l", "duplicate_patient_id"
]
df = df[columns].sort_values(["hospital", "patient_id"], na_position="last").reset_index(drop=True)
df.to_csv(OUTPUT_DIR / "patients_harmonises.csv", index=False, date_format="%Y-%m-%d")

schema_rows = [
    ("patient_id", "texte", "Identifiant d'origine; l'unicité inter-hôpitaux n'est pas garantie"),
    ("hospital", "catégorie A/B/C", "Établissement source"),
    ("admission_date", "date ISO", "Date d'admission harmonisée"),
    ("discharge_date", "date ISO", "Date de sortie harmonisée"),
    ("age_years", "années", "Âge; hors 18–110 mis manquant"),
    ("sex", "M/F", "Sexe harmonisé; autres modalités manquantes"),
    ("weight_kg", "kg", "Poids converti en kilogrammes"),
    ("height_cm", "cm", "Taille convertie en centimètres"),
    ("bmi_kg_m2", "kg/m²", "IMC calculé à partir du poids et de la taille"),
    ("smoking", "0/1", "Tabagisme : 0 non, 1 oui, NA inconnu"),
    ("diabetes", "0/1", "Diabète : 0 non, 1 oui, NA inconnu"),
    ("hypertension", "0/1", "HTA : 0 non, 1 oui, NA inconnu"),
    ("icu", "0/1", "Passage en soins intensifs"),
    ("los_days", "jours", "Durée calculée : date de sortie moins date d'admission"),
    ("los_reported_days", "jours", "Durée fournie par la source, conservée pour audit"),
    ("readmit30", "0/1", "Réhospitalisation à 30 jours selon la source"),
    ("in_hospital_death", "0/1", "Décès durant l'hospitalisation"),
    ("diagnosis_raw", "texte", "Diagnostic original conservé"),
    ("sbp_mmhg", "mmHg", "Pression artérielle systolique"),
    ("creatinine_mg_dl", "mg/dL", "Créatinine"),
    ("crp_mg_l", "mg/L", "CRP"),
    ("duplicate_patient_id", "booléen", "Identifiant répété dans le fichier combiné"),
]
schema = pd.DataFrame(schema_rows, columns=["variable", "type_unite", "definition_regle"])
schema.to_csv(OUTPUT_DIR / "schema_cible.csv", index=False)
display(schema)
print("CSV harmonisé et schéma cible enregistrés dans :", OUTPUT_DIR.resolve())

,variable,type_unite,definition_regle
0,patient_id,texte,Identifiant d'origine; l'unicité inter-hôpitau...
1,hospital,catégorie A/B/C,Établissement source
2,admission_date,date ISO,Date d'admission harmonisée
3,discharge_date,date ISO,Date de sortie harmonisée
4,age_years,années,Âge; hors 18–110 mis manquant
5,sex,M/F,Sexe harmonisé; autres modalités manquantes
6,weight_kg,kg,Poids converti en kilogrammes
7,height_cm,cm,Taille convertie en centimètres
8,bmi_kg_m2,kg/m²,IMC calculé à partir du poids et de la taille
9,smoking,0/1,"Tabagisme : 0 non, 1 oui, NA inconnu"


CSV harmonisé et schéma cible enregistrés dans : C:\Users\HP\Downloads\sujet (1)\sujet\resultats


## 6. Profil de qualité et données manquantes

In [15]:
quality = pd.DataFrame({
    "variable": df.columns,
    "n_manquants": [int(df[c].isna().sum()) for c in df.columns],
    "pct_manquants": [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
})
quality.to_csv(OUTPUT_DIR / "profil_qualite.csv", index=False)
display(quality.sort_values("pct_manquants", ascending=False))
print("Aucune imputation générale n'a été effectuée : les valeurs inconnues restent manquantes.")

,variable,n_manquants,pct_manquants
3,discharge_date,2526,25.26
2,admission_date,2526,25.26
13,los_days,2526,25.26
8,bmi_kg_m2,821,8.21
6,weight_kg,454,4.54
18,sbp_mmhg,434,4.34
5,sex,415,4.15
20,crp_mg_l,410,4.10
17,diagnosis_raw,405,4.05
14,los_reported_days,402,4.02


Aucune imputation générale n'a été effectuée : les valeurs inconnues restent manquantes.


## 7. Analyse descriptive de la réhospitalisation à 30 jours

In [16]:
valid = df[df["readmit30"].isin([0, 1])].copy()

def rate_table(data, group_col):
    if data.empty:
        return pd.DataFrame()
    return data.groupby(group_col, dropna=False)["readmit30"].agg(
        n="count", readmissions="sum", taux="mean"
    ).assign(taux_pct=lambda x: (x["taux"] * 100).round(2))

overall_rate = float(valid["readmit30"].mean() * 100) if len(valid) else np.nan
print(f"Nombre total de lignes : {len(df)}")
print(f"Issue réhospitalisation renseignée : {len(valid)}")
print(f"Taux observé de réhospitalisation : {overall_rate:.2f} %" if pd.notna(overall_rate) else "Taux non estimable")

print("\nTaux par hôpital")
display(rate_table(valid, "hospital"))
print("\nTaux selon diabète (0 = non, 1 = oui)")
display(rate_table(valid[valid["diabetes"].isin([0, 1])], "diabetes"))
print("\nTaux selon passage en soins intensifs")
display(rate_table(valid[valid["icu"].isin([0, 1])], "icu"))

print("\nDurée de séjour selon réhospitalisation")
los_summary = valid.dropna(subset=["los_days"]).groupby("readmit30")["los_days"].agg(
    n="count", moyenne="mean", mediane="median"
).round(2)
display(los_summary)

core = valid.dropna(subset=["age_years", "sex", "diabetes", "los_days"])
print(f"\nCas complets pour âge, sexe, diabète et durée : {len(core)} / {len(valid)} issues observées")
if len(core):
    print(f"Taux chez les cas complets : {core['readmit30'].mean()*100:.2f} %")
print("Comparer ce taux au taux global aide à voir si les données manquantes peuvent changer la description.")

Nombre total de lignes : 10000
Issue réhospitalisation renseignée : 9603
Taux observé de réhospitalisation : 8.38 %

Taux par hôpital


,n,readmissions,taux,taux_pct
hospital,,,,
A,3816,317.0,0.083071,8.31
B,3357,299.0,0.089068,8.91
C,2430,189.0,0.077778,7.78



Taux selon diabète (0 = non, 1 = oui)


,n,readmissions,taux,taux_pct
diabetes,,,,
0.0,7603,612.0,0.080495,8.05
1.0,2000,193.0,0.096500,9.65



Taux selon passage en soins intensifs


,n,readmissions,taux,taux_pct
icu,,,,
0.0,8209,667.0,0.081252,8.13
1.0,1013,96.0,0.094768,9.48



Durée de séjour selon réhospitalisation


,n,moyenne,mediane
readmit30,,,
0.0,6596,12.12,4.0
1.0,614,9.78,4.0



Cas complets pour âge, sexe, diabète et durée : 6739 / 9603 issues observées
Taux chez les cas complets : 8.58 %
Comparer ce taux au taux global aide à voir si les données manquantes peuvent changer la description.


## 8. Régression logistique exploratoire (facultative)

In [17]:
model_result = {"note": "Modèle exploratoire; association statistique ne signifie pas causalité."}
if STATSMODELS_AVAILABLE:
    model_data = valid.dropna(subset=["age_years", "diabetes", "los_days", "sex", "icu"]).copy()
    model_data["sex_M"] = (model_data["sex"] == "M").astype(int)
    predictors = ["age_years", "diabetes", "los_days", "sex_M", "icu"]
    if len(model_data) >= 50 and model_data["readmit30"].nunique() == 2:
        try:
            X = sm.add_constant(model_data[predictors].astype(float), has_constant="add")
            fit = sm.Logit(model_data["readmit30"].astype(float), X).fit(disp=False)
            ci = fit.conf_int()
            model_result = {
                "n_cas_complets": int(len(model_data)),
                "odds_ratios": {k: round(float(np.exp(v)), 3) for k, v in fit.params.items()},
                "IC95_OR": {k: [round(float(np.exp(a)), 3), round(float(np.exp(b)), 3)] for k, (a, b) in ci.iterrows()},
                "p_values": {k: round(float(v), 5) for k, v in fit.pvalues.items()},
                "note": "OR ajustés exploratoires sur cas complets; modèle non causal et potentiellement limité par les données manquantes."
            }
        except Exception as exc:
            model_result = {"non_estime": str(exc)}
    else:
        model_result = {"non_estime": "Effectif complet inférieur à 50 ou issue sans variation."}
else:
    model_result = {"non_estime": "Installer statsmodels pour lancer ce modèle."}

print(json.dumps(model_result, ensure_ascii=False, indent=2))


{
  "n_cas_complets": 6501,
  "odds_ratios": {
    "const": 0.106,
    "age_years": 0.997,
    "diabetes": 1.205,
    "los_days": 0.997,
    "sex_M": 1.043,
    "icu": 1.064
  },
  "IC95_OR": {
    "const": [
      0.075,
      0.148
    ],
    "age_years": [
      0.992,
      1.002
    ],
    "diabetes": [
      0.98,
      1.482
    ],
    "los_days": [
      0.994,
      1.001
    ],
    "sex_M": [
      0.875,
      1.243
    ],
    "icu": [
      0.805,
      1.408
    ]
  },
  "p_values": {
    "const": 0.0,
    "age_years": 0.24885,
    "diabetes": 0.07693,
    "los_days": 0.13844,
    "sex_M": 0.63955,
    "icu": 0.66175
  },
  "note": "OR ajustés exploratoires sur cas complets; modèle non causal et potentiellement limité par les données manquantes."
}


## 9. Exporter les résultats et générer un rapport court

In [18]:
missing_pct = {c: round(float(df[c].isna().mean() * 100), 2) for c in
               ["readmit30", "age_years", "sex", "diabetes", "los_days", "smoking", "icu",
                "sbp_mmhg", "creatinine_mg_dl", "crp_mg_l"]}
by_hospital = {}
for h, g in valid.groupby("hospital"):
    by_hospital[str(h)] = {"n": int(len(g)), "taux_pct": round(float(g["readmit30"].mean()*100), 2)}
by_diabetes = {}
for k, g in valid[valid["diabetes"].isin([0, 1])].groupby("diabetes"):
    by_diabetes[str(int(k))] = {"n": int(len(g)), "readmissions": int(g["readmit30"].sum()),
                                "taux_pct": round(float(g["readmit30"].mean()*100), 2)}
los_by_readmit = {}
for k, g in valid.dropna(subset=["los_days"]).groupby("readmit30"):
    los_by_readmit[str(int(k))] = {"n": int(len(g)), "moyenne_jours": round(float(g["los_days"].mean()), 2),
                                   "mediane_jours": round(float(g["los_days"].median()), 2)}

results = {
    "effectif_total": int(len(df)),
    "effectifs_par_hopital": {str(k): int(v) for k, v in df["hospital"].value_counts().sort_index().items()},
    "lignes_identifiant_duplique": int(df["duplicate_patient_id"].sum()),
    "valeurs_hors_bornes_mises_manquantes": invalid_counts,
    "dates_incoherentes_ou_manquantes": date_incoherences,
    "manquants_pct": missing_pct,
    "issues_observees_n": int(len(valid)),
    "taux_rehospitalisation_pct": round(overall_rate, 2) if pd.notna(overall_rate) else None,
    "taux_par_hopital": by_hospital,
    "taux_selon_diabete": by_diabetes,
    "duree_selon_rehospitalisation": los_by_readmit,
    "modele_logistique": model_result,
}
(OUTPUT_DIR / "resultats_analyse.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
)

def pretty(x):
    return json.dumps(x, ensure_ascii=False)

report = f'''# TP FIA4 — Harmonisation et qualité des données de santé

## 1. Objectif et sources
Les trois fichiers hospitaliers synthétiques A, B et C ont été harmonisés pour étudier descriptivement la réhospitalisation à 30 jours. Les données sont pédagogiques et ne représentent pas des patients réels.

Effectif combiné : **{len(df)} lignes**. Les effectifs par hôpital sont : {pretty(results["effectifs_par_hopital"])}.

## 2. Nettoyage et harmonisation
- Noms de variables harmonisés vers un schéma commun.
- Dates parsées en dates standard; les dates manquantes ou les sorties antérieures à l'admission sont invalidées.
- Poids du fichier B convertis de livres en kg; tailles converties en cm lorsque l'unité source l'indique ou selon la règle documentée pour C.
- Sexe harmonisé en M/F; variables binaires harmonisées en 0/1. Les modalités inconnues deviennent manquantes.
- Valeurs hors des plages de contrôle documentées dans le notebook mises en NA, sans imputation arbitraire.
- Durée de séjour recalculée à partir des dates; durée source conservée séparément.
- Identifiants répétés signalés, sans suppression automatique.
- Diagnostic source conservé; les codes ou libellés repérés peuvent compléter diabète/HTA seulement si l'indicateur dédié manque.
- Les valeurs manquantes restent NA; le profil détaillé est dans `profil_qualite.csv`.

Valeurs hors bornes mises en manquantes : {pretty(invalid_counts)}. Dates incohérentes ou manquantes : {date_incoherences}.

## 3. Schéma cible et fiabilité
Le dictionnaire est fourni dans `schema_cible.csv`. Les variables binaires, l'âge, le sexe et les dates peuvent être exploités après vérification des taux de manquants et de la cohérence par établissement. Le diagnostic reste hétérogène. La durée calculée dépend de la qualité des dates. La variable `readmit30` est reprise de la source; sa définition opérationnelle et l'exhaustivité du suivi ne peuvent pas être validées à partir des seuls CSV.

## 4. Analyse descriptive
- Issues de réhospitalisation renseignées : **{len(valid)}**.
- Taux global observé : **{overall_rate:.2f}%** si estimable.
- Taux par hôpital : {pretty(by_hospital)}.
- Taux selon diabète (0 = non, 1 = oui) : {pretty(by_diabetes)}.
- Durée selon réhospitalisation : {pretty(los_by_readmit)}.
- Modèle logistique exploratoire : {pretty(model_result)}.

Ces résultats sont des associations descriptives, pas des effets causaux. Le modèle ne porte que sur les cas complets et peut être biaisé si les données manquent de façon non aléatoire.

## 5. Limites et conclusion
Les hôpitaux peuvent différer par les populations prises en charge, les périodes, les codages et les définitions de la réhospitalisation. Les différences observées peuvent donc refléter des effets de structure. Avant toute conclusion confirmatoire, il faudrait valider les définitions avec les établissements, vérifier les doublons et documenter les mécanismes de données manquantes. Aucune généralisation clinique ne doit être faite à partir de ces données synthétiques.
'''
(OUTPUT_DIR / "rapport_fia4.md").write_text(report, encoding="utf-8")

print("Tous les livrables ont été enregistrés dans :", OUTPUT_DIR.resolve())
print("\nContenu du dossier resultats :")
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)


Tous les livrables ont été enregistrés dans : C:\Users\HP\Downloads\sujet (1)\sujet\resultats

Contenu du dossier resultats :
- patients_harmonises.csv
- profil_qualite.csv
- rapport_fia4.md
- resultats_analyse.json
- schema_cible.csv
